In [73]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [74]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/MCG+01-32-049_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/MCG+01-32-049_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     377   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        58   (2094, 964)   float32   


In [75]:
image_cut = image[52:172, 0:1890]
image_error_cut = image_error[52:172, 0:1890]

In [76]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00309
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [77]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [78]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049\MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/PSF/Real_seeing_MGC+01-32-049.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 120, 1)

In [79]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]    

## Halpha

In [80]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))

sky_err = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Propag_errori/Sky_errors.txt")
real_errors_HA = np.sqrt(radial_profile_error_HA**2+(sky_err[2])**2+(0.04*radial_profile_HA)**2)

fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/Halpha_fit.csv", index_col=0)

In [81]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.1330749867078596
1.6441797529293292
0.8458781390160057
0.8420459810754362
35.10550574347173
25.258012424177487
0.5444381643776929
0.5541519125246646
1.565531107783728
1.593451042091355
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       4.630665e+01  8.910700e-02  5.687778e-05   0.000252   0.000061
Green      9.681572e+00  5.266100e+01  0.000000e+00   0.000000   0.000000
Purple     2.493663e+01  3.345865e+01  9.913413e+01   7.101011   1.000247
Orange     1.907515e+01  1.379125e+01  8.658137e-01  90.767414  17.497082
Chocolate  6.287635e-12  4.948083e-11  4.419518e-09   2.131322  81.502610


In [82]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.129147200838127
2.883300614373746
2.5210612425223484
2.5075396922412567
58.12835583399633
43.441931674414015
1.6107307273832552
1.6357363142173322
5.609221596164578
5.671055508433014
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       3.691133e+01  2.375100e-01  6.940648e-05   0.000256   0.000069
Green      1.590707e+01  5.188983e+01  0.000000e+00   0.000000   0.000000
Purple     2.687411e+01  3.398881e+01  9.895365e+01   7.225797   1.176547
Orange     2.030749e+01  1.388385e+01  1.046283e+00  90.393621  20.047847
Chocolate  7.729064e-12  5.381986e-11  6.204980e-09   2.380327  78.775537


In [83]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

3.0854854672411713
3.896805595389216
4.763698908714648
4.8945354561681125
70.53506833040787
54.54708309745028
3.101496680878551
3.1300417740000195
7.36195047098205
7.429699414058988
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       2.777868e+01  3.623488e+00  8.729221e-05   0.000261   0.000079
Green      2.278925e+01  4.741251e+01  0.000000e+00   0.000000   0.000000
Purple     2.840372e+01  3.438754e+01  9.870551e+01   7.220076   1.356182
Orange     2.102835e+01  1.457647e+01  1.294404e+00  87.863715  23.025314
Chocolate  1.001238e-11  5.877184e-11  9.691015e-09   4.915948  75.618425


## HBeta

In [84]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
real_errors_HB = np.sqrt(radial_profile_error_HB**2+(sky_err[2])**2+(0.04*radial_profile_HB)**2)
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/Hbeta_fit.csv", index_col = 0)

In [85]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.566251806342246
0.5650321070332929
0.7377674587410908
0.7369524791990527
3.3851216674853095
3.706560771180502
1.0090977608335179
0.9982612095268573
0.4585108913445809
0.4983233776557709
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       3.594277e+01  4.194607e-01  0.000000e+00   0.000000   0.000000
Green      2.437507e+01  4.742555e+01  2.017905e-48   0.000000   0.000000
Purple     2.379770e+01  3.749865e+01  9.727155e+01   3.674664   1.779431
Orange     1.588446e+01  1.465634e+01  2.728447e+00  96.297491  57.110146
Chocolate  3.605537e-15  3.928435e-14  7.809744e-12   0.027845  41.110423


In [86]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.0995326241745127
1.0973012741203063
1.4906442163950946
1.4890760246671721
9.62468164602841
10.455137891929196
1.9541784757057785
1.938135522840284
1.3261294929163236
1.4288701110746451
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       3.236305e+01  1.073643e+00  0.000000e+00   0.000000   0.000000
Green      2.648871e+01  4.670757e+01  9.466487e-18   0.000000   0.000000
Purple     2.477478e+01  3.769880e+01  9.711949e+01   3.818844   1.850053
Orange     1.637346e+01  1.451999e+01  2.880509e+00  96.139552  59.332381
Chocolate  4.241210e-15  4.468804e-14  8.986524e-12   0.041603  38.817566


In [87]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5810278357826553
1.5775999296579004
2.270281840543109
2.268421908110561
15.656926915723634
16.62338953873506
2.819089596508353
2.8015976220077032
2.401655322679964
2.52188767927603
                 Peak 1        Peak 2         Peak 3     Peak 4     Peak 5
Blue       2.758799e+01  2.380749e+00  1.942610e-293   0.000000   0.000000
Green      2.899831e+01  4.518825e+01   5.500449e-06   0.000000   0.000000
Purple     2.631114e+01  3.811042e+01   9.652379e+01   4.012437   2.161278
Orange     1.710257e+01  1.432058e+01   3.476203e+00  95.911803  68.181560
Chocolate  5.449699e-15  5.471866e-14   1.160203e-11   0.075760  29.657162


## NII

In [88]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/NII_fit.csv", index_col = 0)

In [89]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.11744146594925138
0.14886810531127803
0.33651269003580575
0.33897971570110946
0.834493020575824
0.901199841800115
0.4606207207161489
0.461432159082823
0.29923179385406157
0.3971582593134677
              Peak 1     Peak 2        Peak 3        Peak 4        Peak 5
Blue       24.799436   0.364081  1.003260e-05  1.242280e-09  2.120374e-10
Green       0.070105  25.878796  6.592083e-22  4.146811e-57  1.964530e-65
Purple     36.261433  41.158030  8.877241e+01  3.423634e+00  2.897562e+00
Orange     38.865050  32.595640  1.122617e+01  9.650686e+01  8.716292e+01
Chocolate   0.003977   0.003453  1.417795e-03  6.950244e-02  9.939517e+00


In [90]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.4648967987640921
0.5555246326514736
0.6351726792462923
0.6401101541795663
2.3969862877028003
2.569574407656669
1.7966081226808177
1.7990029559633909
0.8903473813472211
1.145874973649732
              Peak 1     Peak 2        Peak 3        Peak 4        Peak 5
Blue       19.673995   0.877423  1.106759e-05  1.162676e-09  2.167815e-10
Green       1.409286  20.500637  9.462625e-21  8.983384e-57  4.851109e-65
Purple     38.799700  44.046911  8.826672e+01  3.399265e+00  2.924922e+00
Orange     40.112899  34.571364  1.173179e+01  9.651557e+01  8.799131e+01
Chocolate   0.004121   0.003665  1.482910e-03  8.516786e-02  9.083768e+00


In [91]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6862852989204943
0.7884203691196857
0.9177383383334367
0.9314703056084108
4.093845320429202
4.299052554309666
2.616347503505755
2.6189370062478687
2.021413033589276
2.3346436672501456
              Peak 1     Peak 2        Peak 3        Peak 4        Peak 5
Blue       15.180329   2.095451  1.304468e-05  1.266352e-09  2.443596e-10
Green       4.111410  15.449590  4.160147e-19  6.104525e-56  1.527268e-63
Purple     39.882483  46.503006  8.591182e+01  3.520486e+00  3.029925e+00
Orange     40.821580  35.948137  1.408637e+01  9.638366e+01  9.119939e+01
Chocolate   0.004197   0.003816  1.805873e-03  9.585069e-02  5.770690e+00


## SII

In [92]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/SII_fit.csv", index_col = 0)

In [93]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.38862320006562423
0.3884499488001907
0.21958610751729077
0.2194990816213494
1.0457099129693381
1.1117287458708551
0.47914181561713876
0.4800372280215855
0.43652589817917736
0.43723387063110897
              Peak 1     Peak 2         Peak 3        Peak 4     Peak 5
Blue       19.895000   5.460762   2.476225e-94  0.000000e+00   0.000000
Green      19.939345  25.382899  8.521192e-218  0.000000e+00   0.000000
Purple     28.598865  37.464511   8.910377e+01  3.771975e+00   2.207753
Orange     31.566789  31.691828   1.089623e+01  9.622802e+01  65.080319
Chocolate   0.000000   0.000000   0.000000e+00  5.109054e-07  32.711928


In [94]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.758967237454754
0.7583467959626196
0.878152563886037
0.8779666370778694
3.006794692178553
3.1768849250520588
1.8718060390549298
1.8744506672059944
1.2817747108637476
1.2835933893213816
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       18.903884   8.173598  1.712113e-66   0.000000   0.000000
Green      19.241537  25.003121  9.488989e-59   0.000000   0.000000
Purple     29.503919  35.670233  8.862606e+01   3.733837   2.258546
Orange     32.350660  31.153047  1.137394e+01  96.265434  66.562039
Chocolate   0.000000   0.000000  0.000000e+00   0.000729  31.179415


In [95]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

1.1040009065044534
1.1028294436856347
1.3161303064471015
1.315787277630387
5.116048425503231
5.311938657205563
2.73294489033244
2.735797313464864
2.4131620000809266
2.414890911083316
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       17.101352   8.466391  1.425528e-30   0.000000   0.000000
Green      18.695239  24.036069  4.183759e-07   0.000000   0.000000
Purple     30.802638  36.279044  8.686521e+01   3.858566   2.493525
Orange     33.400771  31.218496  1.313479e+01  96.133347  72.985251
Chocolate   0.000000   0.000000  0.000000e+00   0.008087  24.521224


## OIII

In [96]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/OIII_fit.csv", index_col = 0)

In [97]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5582164039191169
1.5029057348986714
1.049538893881619
1.04889204893338
12.879486825167389
14.30844240027823
1.230100713650939
1.2174652405093973
0.6814577450737419
0.6743725826931537
              Peak 1     Peak 2        Peak 3     Peak 4        Peak 5
Blue       64.401128   3.151890  9.992474e-04   0.000002  1.684040e-07
Green      15.394060  63.384902  5.845510e-16   0.000000  0.000000e+00
Purple     12.891077  26.821276  9.909502e+01   3.035615  1.035275e+00
Orange      7.313735   6.641933  9.039759e-01  96.963967  5.095301e+01
Chocolate   0.000000   0.000000  0.000000e+00   0.000416  4.801171e+01


In [98]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.9553067669751347
2.8723263823163636
3.149538646791389
3.1478366342923865
36.33208099503895
40.024585448773685
2.3839268642204425
2.365237570540122
1.9827141142620055
1.9594169016813665
              Peak 1     Peak 2        Peak 3     Peak 4        Peak 5
Blue       57.918772   3.400848  1.100394e-03   0.000002  1.770384e-07
Green      20.533805  62.796703  2.185410e-10   0.000000  0.000000e+00
Purple     13.829039  27.159676  9.903701e+01   3.165881  1.071904e+00
Orange      7.718385   6.642773  9.618915e-01  96.827660  5.262896e+01
Chocolate   0.000000   0.000000  0.000000e+00   0.006458  4.629913e+01


In [99]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

4.215240094413153
4.123153189596946
6.168356337113691
6.161216727275197
62.293189130537634
66.66148233323737
3.4432334045531596
3.422771486990275
3.9367620404377313
3.887497476925465
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       50.166577   6.630953   0.001880   0.000002  2.363395e-07
Green      26.740396  60.267312   0.000008   0.000000  0.000000e+00
Purple     14.965633  26.431176  98.684262   3.345978  1.295165e+00
Orange      8.127395   6.670559   1.313850  96.588549  6.259867e+01
Chocolate   0.000000   0.000000   0.000000   0.065471  3.610617e+01
